# Обратное распространение ошибки (backpropagation)

**Backpropagation** — способ эффективно вычислить градиент функции потерь по всем параметрам модели. Он применяет цепное правило от выхода вычислительного графа к его входам и повторно использует уже найденные производные.

Конспект основан на презентации Филиппа Ульянкина [«Посиделка 4: Алгоритм обратного распространения ошибки»](https://github.com/FUlyankin/deep_learning_pytorch/blob/main/week04_backprop/nn_slides_4.pdf). 

## Содержание

1. Зачем нужен backpropagation
2. Цепное правило и несколько путей зависимости
3. Вычислительный граф: два прохода
4. Как работает обратный проход
5. Один нейрон: вывод градиентов
6. Ручной численный пример
7. Полносвязный слой: матричные формулы
8. Как соединить слои
9. Почему алгоритм эффективен
10. Проверка производных и типичные ошибки
11. Что нужно запомнить


## 1. Зачем нужен backpropagation

В линейной регрессии прогноз задаётся одной простой формулой. Нейросеть — композиция преобразований: выход одного слоя становится входом следующего. Но задача обучения остаётся прежней:

$$
\hat y = F_\theta(x), \qquad L = \ell(\hat y,y), \qquad
\theta^+ = \theta - \eta\nabla_\theta L.
$$

Здесь $\theta$ — все обучаемые веса и смещения, $y$ — правильный ответ, $\eta>0$ — шаг обучения. Чтобы обновить параметры, нужно узнать, как каждый из них влияет на итоговые потери.

| Этап | Что вычисляем |
|---|---|
| Прямой проход (forward) | Промежуточные значения, прогноз и потери |
| Обратный проход (backward) | Производные потерь по промежуточным значениям и параметрам |
| Шаг оптимизатора | Новые параметры на основе найденных градиентов |

**Backpropagation вычисляет градиенты; градиентный спуск использует их для обновления весов.** Это разные этапы. Один и тот же градиент можно передать разным оптимизаторам.


## 2. Цепное правило и несколько путей зависимости

Если $L=f(z)$, а $z=g(x)$, то

$$
\frac{dL}{dx}=\frac{dL}{dz}\frac{dz}{dx}.
$$

Первый множитель показывает чувствительность потерь к $z$, второй — чувствительность $z$ к $x$. Их произведение связывает изменение входа с изменением потерь.

Если $x$ влияет на результат через **несколько ветвей**, их вклады складываются:

$$
z_1=g_1(x),\quad z_2=g_2(x),\quad L=f(z_1,z_2),
$$

$$
\boxed{
\frac{dL}{dx}
=\frac{\partial L}{\partial z_1}\frac{dz_1}{dx}
+\frac{\partial L}{\partial z_2}\frac{dz_2}{dx}
}
$$

Например, для $z_1=x^2$, $z_2=3x$ и $L=z_1+z_2$ получим $dL/dx=2x+3$.

**Вдоль одного пути производные перемножаются, вклады разных путей суммируются.** Это правило понадобится и при разветвлении графа, и при многократном использовании одного параметра.


## 3. Вычислительный граф: два прохода

Сложную формулу удобно разбить на элементарные операции. Каждая операция становится узлом графа, а зависимости между значениями — его рёбрами.

![Прямой и обратный проходы в графе с двумя ветвями](assets/backprop_graph.svg)

В прямом проходе вычисляем значения в порядке зависимостей и сохраняем то, что понадобится для производных. Например, для обратного прохода через сигмоиду удобно сохранить её выход.

В обратном проходе начинаем со скалярной функции потерь:

$$
\frac{\partial L}{\partial L}=1.
$$

Затем двигаемся в обратном порядке. Для каждой операции достаточно знать её локальные производные и градиент, который пришёл со стороны выхода.

Через граф назад передаётся **чувствительность потерь**, а не сам правильный ответ и не одна и та же величина $\hat y-y$ на всех слоях.


## 4. Как работает обратный проход

Обозначим $g_u=\partial L/\partial u$. Для операции $v=f(u)$ её вклад в градиент входа равен

$$
g_u \mathrel{+}= g_v\frac{\partial v}{\partial u}.
$$

Знак $\mathrel{+}=$ важен: если значение $u$ используется в нескольких местах, нужно накопить вклад **от каждого** использования.

| Операция | Вклад в градиенты входов |
|---|---|
| $v=a+b$ | $g_a\mathrel{+}=g_v$, $g_b\mathrel{+}=g_v$ |
| $v=ab$ | $g_a\mathrel{+}=g_vb$, $g_b\mathrel{+}=g_va$ |
| $v=\sigma(a)$ | $g_a\mathrel{+}=g_vv(1-v)$ |

Все величины в этой таблице — скаляры. Для матриц правила учитывают размерности и порядок умножения.

Общий порядок действий:

1. Выполнить forward и сохранить нужные промежуточные значения.
2. Начать backward с градиента потерь, равного единице.
3. Обойти операции в обратном порядке зависимостей, суммируя вклады ветвей.
4. Получить градиенты параметров.
5. После завершения backward обновить параметры оптимизатором.

Нельзя менять вес посередине обратного прохода, если он ещё нужен для других производных: все градиенты этого шага должны соответствовать одному и тому же forward.


## 5. Один нейрон: вывод градиентов

Рассмотрим один объект и нейрон с сигмоидой:

$$
s=\sum_{j=1}^{d}w_jx_j+b,\qquad
p=\sigma(s)=\frac{1}{1+e^{-s}},\qquad
L=\frac12(p-y)^2.
$$

Квадратичная потеря здесь выбрана для наглядности производных. В бинарной классификации с сигмоидой обычно используют кросс-энтропию; она даст другую производную по выходу.

Проходим от потерь назад:

$$
\frac{\partial L}{\partial p}=p-y,
\qquad
\frac{\partial p}{\partial s}=p(1-p).
$$

Введём общий промежуточный градиент:

$$
\delta=\frac{\partial L}{\partial s}
=(p-y)p(1-p).
$$

Тогда все производные по параметрам получаются из него:

$$
\boxed{\frac{\partial L}{\partial w_j}=\delta x_j},
\qquad
\boxed{\frac{\partial L}{\partial b}=\delta}.
$$

Если вход нейрона — выход предыдущего слоя, градиент передаётся дальше:

$$
\frac{\partial L}{\partial x_j}=\delta w_j.
$$

**Общий множитель $\delta$ вычисляется один раз и используется для всех весов.** Это маленький пример того, как backpropagation экономит вычисления.


## 6. Ручной численный пример

Пусть $x_1=2$, $x_2=-1$, $w_1=0.5$, $w_2=1$, $b=0$, правильный ответ $y=1$.

| Шаг | Вычисление | Результат |
|---|---|---:|
| Линейная комбинация | $s=0.5\cdot2+1\cdot(-1)+0$ | $0$ |
| Прогноз | $p=\sigma(0)$ | $0.5$ |
| Потери | $L=\frac12(0.5-1)^2$ | $0.125$ |
| Градиент по выходу | $\partial L/\partial p=p-y$ | $-0.5$ |
| Градиент до сигмоиды | $\delta=-0.5\cdot0.5\cdot0.5$ | $-0.125$ |
| Градиент первого веса | $\partial L/\partial w_1=\delta x_1$ | $-0.25$ |
| Градиент второго веса | $\partial L/\partial w_2=\delta x_2$ | $0.125$ |
| Градиент смещения | $\partial L/\partial b=\delta$ | $-0.125$ |

При шаге $\eta=0.1$ одновременно обновляем параметры:

$$
w_1^+=0.525,\qquad w_2^+=0.9875,\qquad b^+=0.0125.
$$

Новый $s^+=0.075$, прогноз $p^+\approx0.5187$, потери $L^+\approx0.1158$.
Прогноз приблизился к единице, а потери уменьшились.

Первый вес вырос, второй уменьшился: направление изменения зависит и от ошибки, и от знака соответствующего входа. Уменьшение потерь в этом примере не означает, что любой размер шага будет удачным.


## 7. Полносвязный слой: матричные формулы

Для нескольких объектов и нейронов используем матрицы. Объекты расположены **в строках**:

| Обозначение | Размер | Смысл |
|---|---|---|
| $X$ | $B\times d$ | Батч из $B$ объектов с $d$ признаками |
| $W$ | $d\times m$ | Веса $m$ выходных нейронов |
| $b$ | $m$ | Смещения |
| $Z$ | $B\times m$ | Выход линейного преобразования |
| $G=\partial L/\partial Z$ | $B\times m$ | Входящий градиент |

Прямой проход:

$$
Z=XW+\mathbf1_B b^\top.
$$

Обратный проход:

$$
\boxed{\frac{\partial L}{\partial W}=X^\top G},\qquad
\boxed{\frac{\partial L}{\partial X}=GW^\top},\qquad
\boxed{\frac{\partial L}{\partial b}=\sum_{i=1}^{B}G_{i,:}}.
$$

Последняя сумма берётся по объектам; результат имеет ту же форму, что и $b$.

Почему возникает $X^\top G$? Из $Z_{ik}=\sum_jX_{ij}W_{jk}+b_k$ следует:

$$
\frac{\partial L}{\partial W_{jk}}
=\sum_{i=1}^{B}\frac{\partial L}{\partial Z_{ik}}X_{ij}
=\sum_{i=1}^{B}G_{ik}X_{ij}.
$$

Это ровно элемент $(j,k)$ матрицы $X^\top G$. Размерности дают быструю проверку:
$(d\times B)(B\times m)=d\times m$, как у $W$.

Если после линейной части стоит **поэлементная** активация $A=\varphi(Z)$, то

$$
\frac{\partial L}{\partial Z}
=\frac{\partial L}{\partial A}\odot\varphi'(Z),
$$

где $\odot$ — поэлементное умножение. Для сигмоиды $\varphi'(Z)=A\odot(1-A)$.
Эта формула не переносится без изменений на операции, связывающие разные компоненты, например softmax.


## 8. Как соединить слои

Возьмём схему из презентации: два скрытых слоя с сигмоидой и линейный выход. Для краткости опустим смещения:

$$
Z_1=XW_1,\quad A_1=\sigma(Z_1),\quad
Z_2=A_1W_2,\quad A_2=\sigma(Z_2),\quad
\hat Y=A_2W_3.
$$

Зафиксируем потерю, чтобы не путаться в знаках и коэффициентах:

$$
L=\frac{1}{2B}\|\hat Y-Y\|_F^2.
$$

Это половина суммы квадратов ошибок по выходам, усреднённая по $B$ объектам. При одном выходе — половина обычной MSE.

**От выходного слоя к первому:**

$$
G_3=\frac{\partial L}{\partial\hat Y}=\frac{\hat Y-Y}{B},
\qquad
\frac{\partial L}{\partial W_3}=A_2^\top G_3;
$$

$$
G_2=(G_3W_3^\top)\odot A_2\odot(1-A_2),
\qquad
\frac{\partial L}{\partial W_2}=A_1^\top G_2;
$$

$$
G_1=(G_2W_2^\top)\odot A_1\odot(1-A_1),
\qquad
\frac{\partial L}{\partial W_1}=X^\top G_1.
$$

Каждый скрытый слой повторяет два действия: получает градиент через веса следующего слоя и умножает его на производную своей активации.

Коэффициент $1/B$ уже учтён в $G_3$ и проходит дальше по цепочке. **Повторно делить каждый градиент на размер батча не нужно.** Если вернуть смещения, их градиенты будут суммами соответствующих $G_\ell$ по строкам.

> Уточнение к источнику: на слайдах 27–28 в схеме указан множитель $-2(\hat y-y)$. Для потери $(\hat y-y)^2$ производная по $\hat y$ равна **$2(\hat y-y)$**. Знак минус появляется в шаге градиентного спуска. Выше использована согласованная запись с множителем $1/(2B)$ в самой потере.


## 9. Почему алгоритм эффективен

Можно отдельно раскрыть цепное правило для каждого веса и каждого пути до потерь. Но в такой записи многократно повторяются одни и те же подвыражения.

Backpropagation сохраняет промежуточные значения forward и накапливает градиенты узлов backward. Уже вычисленная чувствительность узла используется для всех его входов. Это **автоматическое дифференцирование обратного режима** — reverse-mode autodiff.

Для скалярной функции потерь одного обратного обхода достаточно, чтобы получить производные по множеству параметров. На уровне скалярных операций стоимость пропорциональна размеру вычислительного графа. Для матричных узлов нужно учитывать стоимость их операций; время нельзя оценивать только числом весов, игнорируя размер батча.

Цена такого переиспользования — память для промежуточных значений. Алгоритм получает производные по правилам дифференцирования, а не приближает их малыми возмущениями параметров; при этом обычные погрешности вычислений с плавающей точкой остаются.


## 10. Проверка производных и типичные ошибки

### Численная проверка

Для одного параметра $\theta_j$ градиент можно приблизить центральной разностью:

$$
\frac{\partial L}{\partial\theta_j}
\approx
\frac{L(\theta+\varepsilon e_j)-L(\theta-\varepsilon e_j)}{2\varepsilon},
$$

где $e_j$ меняет только $j$-ю координату. Сравнение с аналитическим градиентом называется **gradient checking**.

Большое $\varepsilon$ даёт грубую аппроксимацию, слишком маленькое усиливает ошибки округления. Для проверки нескольких параметров метод полезен; для обучения большой модели дорог: для $P$ параметров центральная разность требует $2P$ вычислений потерь. Данные и случайные составляющие при сравнении должны быть зафиксированы, а точки недифференцируемости нужно учитывать отдельно.

### На что смотреть при разборе формул

| Ошибка | Что проверить |
|---|---|
| Забыли одну ветвь графа | Учтены ли все пути влияния параметра на потери |
| Перепутали знак | Производная потерь и минус в обновлении весов — разные вещи |
| Лишний коэффициент $B$ | Где именно loss усредняется по батчу |
| Перепутали виды умножения | Матричное произведение и $\odot$ выполняют разные операции |
| Неправильное транспонирование | Форма градиента должна совпадать с формой параметра |
| Изменили веса во время backward | Все производные должны относиться к одному forward |

### Почему правильный backprop ещё не гарантирует хорошее обучение

Производная сигмоиды $p(1-p)\leq1/4$, а в насыщении близка к нулю. Повторное умножение на малые производные может ослабить сигнал для ранних слоёв. Веса тоже входят в произведение: в зависимости от них градиенты могут затухать или расти.

Здесь механизм градиентов помогает понять проблему; сам по себе корректный алгоритм не выбирает удачную архитектуру, инициализацию и шаг обучения.
